# Authenticate and Call M365 Copilot Admin Packages API

This notebook demonstrates how to authenticate with an Entra app registration using delegated access (OAuth) and call the Microsoft 365 Copilot Admin Catalog Packages API.

> **Note:** This API is in **beta** and subject to change. It is not supported for use in production applications.

### 1. Install and Import Required Libraries

In [ ]:
!pip install msal requests

In [ ]:
import msal
import requests
import json
import os
from dotenv import load_dotenv

# Find the .env file by searching up the directory tree
# This makes it independent of where the notebook is run from
current_dir = os.getcwd()
# Go up one level since the notebook is in the 'Copilot' folder
project_root = os.path.dirname(current_dir)
dotenv_path = os.path.join(project_root, '.env')

# Check if the .env file exists at the constructed path
if not os.path.exists(dotenv_path):
    # If not found, assume the CWD is the project root
    dotenv_path = os.path.join(os.getcwd(), '.env')

load_dotenv(dotenv_path=dotenv_path)

### 2. Define Entra App and API Configuration

This notebook loads credentials from your root `.env` file. Make sure it contains the following variables:
- `CopilotAPIPythonClient_Id`
- `CopilotAPIPythonClient_Tenant`

**Note on Permissions:** This notebook uses **Delegated** permissions. In your Entra App Registration, grant the following permissions under **Delegated permissions** for Microsoft Graph:
- `CopilotPackages.ReadWrite.All`

1. In your App Registration, go to the **Authentication** tab.
2. Scroll down to **Advanced settings**.
3. Set the toggle for **Allow public client flows** to **Yes**.
4. Click **Save**.
5. Ensure you also have a **Mobile and desktop applications** platform configured with a redirect URI for `http://localhost`.


In [ ]:
# Load Entra App Registration details from .env file
client_id = os.getenv("CopilotAPIPythonClient_Id")
tenant_id = os.getenv("CopilotAPIPythonClient_Tenant")
authority = f"https://login.microsoftonline.com/{tenant_id}"

# Required delegated permission scope for the Copilot Admin Packages API
# Using ReadWrite.All as it has tenant-wide admin consent already granted
scopes = [
    "https://graph.microsoft.com/CopilotPackages.ReadWrite.All"
]

# Beta API endpoint - list all packages in the tenant catalog
api_base_url = "https://graph.microsoft.com/beta/copilot/admin/catalog/packages"


### 3. Acquire Access Token

This step uses the MSAL library to acquire an access token on behalf of the user. It will first try to get a token from the cache. If no token is cached, it will open a browser window for the user to sign in and consent.

In [ ]:
app = msal.PublicClientApplication(
    client_id,
    authority=authority,
    token_cache=msal.SerializableTokenCache()
)

# Try to get a token from the cache
accounts = app.get_accounts()
result = None
if accounts:
    # Assuming the user wants to use the first account found
    result = app.acquire_token_silent(scopes, account=accounts[0])

# If no cached token, or token is expired, acquire a new one interactively
if not result:
    print("No suitable token in cache. Initiating interactive login.")
    result = app.acquire_token_interactive(scopes=scopes)

if "access_token" in result:
    access_token = result["access_token"]
    print("Access token acquired successfully.")
else:
    print("Could not acquire access token.")
    print(result.get("error"))
    print(result.get("error_description"))
    access_token = None

### 4. Configure Optional OData Query Parameters

The API supports the following optional `$filter` parameters:

| Parameter | Type | Description |
|---|---|---|
| `supportedHosts` | string | Filter by host: `Copilot`, `Outlook`, `Teams`, `M365`, `Word`, `SharePoint`, etc. |
| `elementTypes` | string | Filter by element type: `Bots`, `DeclarativeAgent`, `CustomEngineAgent`, `OfficeAddIns` |
| `lastModifiedDateTime` | datetime | Filter by last updated date/time |

**Examples:**
- Filter by host: `supportedHosts/any(h:h eq 'Copilot')`
- Filter by type: `elementTypes/any(h:h eq 'DeclarativeAgent')`
- Filter by date: `lastModifiedDateTime gt 2026-01-01T00:00:00.0000000Z`

Leave `odata_filter` as `None` to retrieve all packages.

In [ ]:
# Set to None to retrieve all packages, or set a $filter expression
# Examples:
#   odata_filter = "supportedHosts/any(h:h eq 'Copilot')"
#   odata_filter = "elementTypes/any(h:h eq 'DeclarativeAgent')"
#   odata_filter = "lastModifiedDateTime gt 2026-01-01T00:00:00.0000000Z"
odata_filter = None

### 5. Make an Authenticated API Call

We use the acquired access token to make a `GET` request to the Copilot Admin Catalog Packages API. The response will include all packages available in the tenant catalog, subject to any filter specified in the previous cell.

In [ ]:
if access_token:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    # Build the request URL, appending the $filter parameter if provided
    params = {}
    if odata_filter:
        params["$filter"] = odata_filter

    try:
        response = requests.get(api_base_url, headers=headers, params=params)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

        api_response = response.json()
        packages = api_response.get("value", [])
        print(f"API call successful. {len(packages)} package(s) returned.")

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response body: {response.text}")
        api_response = None
        packages = []
    except Exception as err:
        print(f"An error occurred: {err}")
        api_response = None
        packages = []
else:
    print("Cannot make API call without an access token.")
    api_response = None
    packages = []

### 6. Print the API Response

Let's print the full JSON response from the API in a readable format.

In [ ]:
if api_response:
    print(json.dumps(api_response, indent=2))

### 7. Display Packages as a Summary Table

For a cleaner view, display key fields from each package in a formatted table.

In [ ]:
if packages:
    # Print a formatted summary table
    col_widths = {"id": 45, "displayName": 40, "type": 10, "isBlocked": 9, "availableTo": 12, "deployedTo": 12}
    header = (
        f"{'ID':<{col_widths['id']}}  "
        f"{'Display Name':<{col_widths['displayName']}}  "
        f"{'Type':<{col_widths['type']}}  "
        f"{'Blocked':<{col_widths['isBlocked']}}  "
        f"{'Available To':<{col_widths['availableTo']}}  "
        f"{'Deployed To':<{col_widths['deployedTo']}}  "
        f"Supported Hosts"
    )
    separator = "-" * len(header)
    print(header)
    print(separator)
    for pkg in packages:
        hosts = ", ".join(pkg.get("supportedHosts", []))
        print(
            f"{pkg.get('id', ''):<{col_widths['id']}}  "
            f"{pkg.get('displayName', ''):<{col_widths['displayName']}}  "
            f"{pkg.get('type', ''):<{col_widths['type']}}  "
            f"{str(pkg.get('isBlocked', '')):<{col_widths['isBlocked']}}  "
            f"{pkg.get('availableTo', ''):<{col_widths['availableTo']}}  "
            f"{pkg.get('deployedTo', ''):<{col_widths['deployedTo']}}  "
            f"{hosts}"
        )
else:
    print("No packages to display.")